# Data Exploration and Reproducible Split

This notebook audits the Dogs vs. Cats dataset, verifies the existing folder structure, checks image quality, and creates reproducible train, validation, and test split files.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, UnidentifiedImageError

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

records = collect_image_records(DATASET_ROOT, CLASS_NAMES)
from src.utils.reproducibility import set_seed

set_seed(42)
DATASET_ROOT = PROJECT_ROOT / 'data' / 'dogs-vs-cats-classification'
CLASS_NAMES = ['cats', 'dogs']
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'

DATASET_ROOT.exists(), DATASET_ROOT

In [ ]:
records = collect_image_records(DATASET_ROOT, CLASS_NAMES)
records.head(), records['label'].value_counts()

In [ ]:
train_records, val_records, test_records = create_stratified_splits(
    records=records,
    test_size=0.30,
    val_size=0.15,
    random_state=42,
)

split_paths = save_split_csvs(train_records, val_records, test_records, SPLITS_DIR)
split_paths
summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'count': [len(train_records), len(val_records), len(test_records)],
})
summary

In [ ]:
def check_image(path_str):
    try:
        with Image.open(path_str) as image:
            image.verify()
        with Image.open(path_str) as image:
            return image.size, image.mode, None
    except (UnidentifiedImageError, OSError) as exc:
        return None, None, str(exc)

sample_checks = records.sample(n=min(20, len(records)), random_state=42).copy()
sample_checks[['size', 'mode', 'error']] = sample_checks['image_path'].apply(lambda p: pd.Series(check_image(p)))
sample_checks

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, UnidentifiedImageError

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data.split_data import collect_image_records, create_stratified_splits, save_split_csvs
from src.utils.reproducibility import set_seed

set_seed(42)
DATASET_ROOT = PROJECT_ROOT / 'data' / 'dogs-vs-cats-classification'
CLASS_NAMES = ['cats', 'dogs']
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'

DATASET_ROOT.exists(), DATASET_ROOT

In [ ]:
from pathlib import Path

folder_tree = []
for path in sorted(DATASET_ROOT.rglob('*')):
    relative_path = path.relative_to(DATASET_ROOT)
    if len(relative_path.parts) <= 2:
        folder_tree.append(str(relative_path))

folder_tree[:20], len(folder_tree)